<a href="https://colab.research.google.com/github/rajendran-priya/amazon-review-summarization/blob/main/scripts/code_review_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1) Install packages

In [5]:
!pip install -q pandas numpy matplotlib seaborn nltk spacy
!pip install -q transformers sentencepiece accelerate bitsandbytes
!pip install -q rouge-score sacrebleu bert-score
!pip install -q sumy networkx scikit-learn

2) Imports

In [6]:
import os
import re
import math
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import nltk
nltk.download("punkt")
nltk.download("stopwords")

from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import AutoModelForCausalLM

from rouge_score import rouge_scorer
from sacrebleu import sentence_bleu
from bert_score import score as bertscore_score

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

from google.colab import files

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


3) Upload the dataset manually

In [8]:
!ls -lh

total 287M
-rw-r--r-- 1 root root 287M Mar 11 05:04 amazon_reviews.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


In [9]:
DATA_FILE = "/content/amazon_reviews.csv"

4) Load dataset

In [10]:
df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (568454, 10)
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


Step 5 — Define dataset columns

In [11]:
TEXT_COL = "Text"
PRODUCT_COL = "ProductId"
SUMMARY_COL = "Summary"
RATING_COL = "Score"

print("Using columns:")
print("TEXT_COL =", TEXT_COL)
print("PRODUCT_COL =", PRODUCT_COL)
print("SUMMARY_COL =", SUMMARY_COL)
print("RATING_COL =", RATING_COL)

Using columns:
TEXT_COL = Text
PRODUCT_COL = ProductId
SUMMARY_COL = Summary
RATING_COL = Score


Step 6 — Clean review text

In [12]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # remove HTML
    text = re.sub(r"<.*?>", " ", text)

    # remove strange characters
    text = re.sub(r"[^a-zA-Z0-9.,!?;:'\"()\-\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df[TEXT_COL] = df[TEXT_COL].apply(clean_text)

# remove very short reviews
df = df[df[TEXT_COL].str.len() > 30]

print("Dataset after cleaning:", df.shape)

Dataset after cleaning: (568385, 10)


Step 7 — Filter products with multiple reviews and keep products with ≥ 5 reviews.

In [13]:
review_counts = (
    df.groupby(PRODUCT_COL)[TEXT_COL]
    .count()
    .reset_index(name="review_count")
)

review_counts = review_counts.sort_values(
    "review_count",
    ascending=False
)

review_counts.head(10)

,ProductId,review_count
71169,B007JFMH8M,913
42262,B002QWP89S,632
42256,B002QWHJOU,632
42263,B002QWP8H0,632
37897,B0026RQTGE,632
46204,B003B3OOPA,623
28623,B001EO5Q64,567
23308,B0013NUGDE,564
35243,B001RVFERK,564
37793,B0026KNQSA,564


In [14]:
MIN_REVIEWS = 5

valid_products = review_counts[
    review_counts["review_count"] >= MIN_REVIEWS
][PRODUCT_COL]

df_filtered = df[df[PRODUCT_COL].isin(valid_products)]

print("Filtered dataset:", df_filtered.shape)
print("Unique products:", df_filtered[PRODUCT_COL].nunique())

Filtered dataset: (475592, 10)
Unique products: 20409


Step 8 — Combine reviews per product

In [15]:
def join_reviews(texts, max_reviews=20):
    texts = list(texts)[:max_reviews]
    return " ".join(texts)


grouped = (
    df_filtered
    .groupby(PRODUCT_COL)[TEXT_COL]
    .apply(lambda x: join_reviews(x, max_reviews=20))
    .reset_index()
)

grouped = grouped.rename(columns={TEXT_COL: "combined_reviews"})

print("Grouped shape:", grouped.shape)
grouped.head()

Grouped shape: (20409, 2)


,ProductId,combined_reviews
0,0006641040,"These days, when a person says, ""chicken soup""..."
1,7310172001,This product is a very health snack for your p...
2,7310172101,This product is a very health snack for your p...
3,B00002N8SM,I don't know how this product performs with bi...
4,B00004CI84,If this is what the afterlife is going to be l...


Step 9 — Create a smaller experiment dataset

In [16]:
SAMPLE_SIZE = 20

sample_df = grouped.sample(
    min(SAMPLE_SIZE, len(grouped)),
    random_state=42
).reset_index(drop=True)

sample_df.head()

,ProductId,combined_reviews
0,B003VIFGWU,I just tried this new soup for the first time ...
1,B000RLGE16,I drank a Duff Energy Drink - yay. Like the Fl...
2,B000X9CX26,I am an enabler. My two dogs have an insatiabl...
3,B0009F3SAU,This has become one of my favorite teas - it i...
4,B003BI2HK4,"My young daughter has wheat, egg dairy allergi..."


Step 10 — Create reference summaries

In [17]:
summary_grouped = (
    df_filtered
    .groupby(PRODUCT_COL)[SUMMARY_COL]
    .apply(lambda x: " ".join(list(x)[:3]))
    .reset_index()
)

summary_grouped = summary_grouped.rename(
    columns={SUMMARY_COL: "reference_summary"}
)

sample_df = sample_df.merge(
    summary_grouped,
    on=PRODUCT_COL,
    how="left"
)

sample_df.head()

,ProductId,combined_reviews,reference_summary
0,B003VIFGWU,I just tried this new soup for the first time ...,Delicious! I love it! Strange Aftertaste
1,B000RLGE16,I drank a Duff Energy Drink - yay. Like the Fl...,Mmm... beer... FOX merchandise with an animate...
2,B000X9CX26,I am an enabler. My two dogs have an insatiabl...,"My dogs love the Greenies, and I like Amazon's..."
3,B0009F3SAU,This has become one of my favorite teas - it i...,Outstanding Tea It is just OK! Great Immune Bo...
4,B003BI2HK4,"My young daughter has wheat, egg dairy allergi...",Best Allergy Free Cookies! Yum!!!! not easy on...


Step 11 — TextRank summarization

In [18]:
!pip install sumy

In [19]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [20]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer


def textrank_summary(text, sentence_count=3):

    parser = PlaintextParser.from_string(
        text,
        Tokenizer("english")
    )

    summarizer = TextRankSummarizer()

    summary = summarizer(
        parser.document,
        sentence_count
    )

    return " ".join(str(sentence) for sentence in summary)


sample_df["textrank_summary"] = sample_df["combined_reviews"].apply(
    textrank_summary
)

sample_df.head()

,ProductId,combined_reviews,reference_summary,textrank_summary
0,B003VIFGWU,I just tried this new soup for the first time ...,Delicious! I love it! Strange Aftertaste,I guessed perhaps the lemongrass was the culpr...
1,B000RLGE16,I drank a Duff Energy Drink - yay. Like the Fl...,Mmm... beer... FOX merchandise with an animate...,With a rampant craze that spread faster and wi...
2,B000X9CX26,I am an enabler. My two dogs have an insatiabl...,"My dogs love the Greenies, and I like Amazon's...",If you're a consistent buyer of Greenies like ...
3,B0009F3SAU,This has become one of my favorite teas - it i...,Outstanding Tea It is just OK! Great Immune Bo...,i had a terrible cold and a patient of mine ha...
4,B003BI2HK4,"My young daughter has wheat, egg dairy allergi...",Best Allergy Free Cookies! Yum!!!! not easy on...,But if you like crunchy cookies you'll love th...


Step 12 — BART summarization

In [21]:
!pip install -U transformers accelerate sentencepiece

In [22]:
Runtime → Restart runtime

SyntaxError: invalid character '→' (U+2192) (4113064584.py, line 1)

In [ ]:
from transformers import pipeline

bart = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    tokenizer="facebook/bart-large-cnn",
    device=0
)